# 99 — Synthetic Data Generator for Fine-Tuning

## Goal
Use a **strong model** to generate **labeled training examples**
that a **weaker/smaller model** can be fine-tuned on. This is
how teams create domain-specific training data without manual labeling.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Seed examples** | Human-written examples that guide generation |
| **Prompt engineering** | Instructing the LLM to generate diverse data |
| **Quality filtering** | Validate generated examples programmatically |
| **Export formats** | JSONL for fine-tuning (Alpaca/ChatML format) |

## Stack
- `langchain-ollama` — LLM for data generation
- Ollama `qwen3.5:9b` (acts as the "strong" teacher model)

> In production you'd use a stronger model (GPT-4, Claude)
> as teacher and fine-tune a smaller model (Llama 3 8B, Phi-3).

In [ ]:
import os, warnings, json, re, random
from pathlib import Path
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOllama(model="qwen3.5:9b", temperature=0.7)  # Higher temp for diversity
print("All imports OK")

## Step 1 — Define the Task & Seed Examples

We'll generate training data for a **customer support intent
classifier** — mapping messages to intent labels.

In [ ]:
# Target task: classify customer messages into intents
INTENTS = [
    "billing_inquiry",
    "technical_support",
    "account_cancellation",
    "product_feedback",
    "shipping_inquiry",
    "refund_request"
]

# Seed examples: 2 per intent (human-written, high quality)
SEED_EXAMPLES = [
    {"message": "Why was I charged $49.99 this month? I thought my plan was $29.99.",
     "intent": "billing_inquiry"},
    {"message": "Can you explain the charges on my latest invoice?",
     "intent": "billing_inquiry"},
    {"message": "The app keeps crashing whenever I try to upload a photo.",
     "intent": "technical_support"},
    {"message": "I can't log in even after resetting my password three times.",
     "intent": "technical_support"},
    {"message": "I'd like to cancel my subscription effective immediately.",
     "intent": "account_cancellation"},
    {"message": "How do I close my account? I no longer need the service.",
     "intent": "account_cancellation"},
    {"message": "The new dashboard redesign is really confusing and hard to navigate.",
     "intent": "product_feedback"},
    {"message": "I love the dark mode feature you just added!",
     "intent": "product_feedback"},
    {"message": "Where is my order? It was supposed to arrive two days ago.",
     "intent": "shipping_inquiry"},
    {"message": "Can I change the delivery address for order #45123?",
     "intent": "shipping_inquiry"},
    {"message": "I want a full refund — the product arrived damaged.",
     "intent": "refund_request"},
    {"message": "The item doesn't match the description. I'd like my money back.",
     "intent": "refund_request"},
]

print(f"Intents: {len(INTENTS)}")
print(f"Seed examples: {len(SEED_EXAMPLES)}")
for intent in INTENTS:
    count = sum(1 for s in SEED_EXAMPLES if s['intent'] == intent)
    print(f"  {intent}: {count} seeds")

## Step 2 — Generate Synthetic Examples

In [ ]:
def generate_examples(intent: str, seeds: list, n: int = 5) -> list:
    """Generate N new examples for a given intent."""
    seed_text = "\n".join(f'- "{s["message"]}"' for s in seeds if s["intent"] == intent)
    
    msg = llm.invoke([
        SystemMessage(content=f"""You are generating training data for an intent classifier.
Generate {n} NEW and DIVERSE customer messages for the intent: "{intent}"

Rules:
- Each message should be realistic, 1-2 sentences
- Vary tone: polite, frustrated, neutral, confused
- Vary specifics: different products, amounts, scenarios
- Do NOT copy the seed examples

Return a JSON array of strings:
["message 1", "message 2", ...]

Return ONLY the JSON array."""),
        HumanMessage(content=f"Seed examples for '{intent}':\n{seed_text}")
    ])
    
    raw = msg.content
    if "<think>" in raw:
        raw = raw.split("</think>")[-1].strip()
    if "```" in raw:
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()
    
    try:
        start = raw.find("[")
        end = raw.rfind("]") + 1
        messages = json.loads(raw[start:end])
    except (json.JSONDecodeError, ValueError):
        messages = []
    
    return [{"message": m, "intent": intent} for m in messages if isinstance(m, str)]

print("Generator function defined")

In [ ]:
# Generate 5 synthetic examples per intent
all_synthetic = []
N_PER_INTENT = 5

for intent in INTENTS:
    print(f"\n🔄 Generating {N_PER_INTENT} examples for '{intent}'...")
    examples = generate_examples(intent, SEED_EXAMPLES, n=N_PER_INTENT)
    print(f"   Generated {len(examples)} examples")
    for ex in examples:
        print(f"   • {ex['message'][:70]}")
    all_synthetic.extend(examples)

print(f"\nTotal synthetic examples: {len(all_synthetic)}")

## Step 3 — Quality Filtering

In [ ]:
def validate_example(example: dict) -> tuple[bool, str]:
    """Validate a synthetic example passes quality checks."""
    msg = example["message"]
    
    # Check 1: Not empty or too short
    if len(msg.strip()) < 10:
        return False, "too_short"
    
    # Check 2: Not too long (fine-tuning data should be concise)
    if len(msg) > 300:
        return False, "too_long"
    
    # Check 3: Valid intent
    if example["intent"] not in INTENTS:
        return False, "invalid_intent"
    
    # Check 4: Not a duplicate of seeds
    for seed in SEED_EXAMPLES:
        if msg.lower().strip() == seed["message"].lower().strip():
            return False, "duplicate_of_seed"
    
    # Check 5: Not just the intent name repeated
    if msg.lower().replace("_", " ") == example["intent"].replace("_", " "):
        return False, "intent_as_message"
    
    return True, "passed"

# Filter
valid = []
rejected = []
for ex in all_synthetic:
    ok, reason = validate_example(ex)
    if ok:
        valid.append(ex)
    else:
        rejected.append({**ex, "reason": reason})

print(f"Validated: {len(valid)} passed, {len(rejected)} rejected")
if rejected:
    for r in rejected:
        print(f"  ❌ [{r['reason']}] {r['message'][:60]}")

## Step 4 — Export for Fine-Tuning

In [ ]:
# Combine seeds + synthetic for the final dataset
final_dataset = SEED_EXAMPLES + valid
random.shuffle(final_dataset)

print(f"Final dataset: {len(final_dataset)} examples")
for intent in INTENTS:
    count = sum(1 for d in final_dataset if d['intent'] == intent)
    print(f"  {intent}: {count} examples")

In [ ]:
# Export as Alpaca-format JSONL (for fine-tuning)
def to_alpaca_format(example):
    return {
        "instruction": "Classify the following customer message into one of these intents: "
                       + ", ".join(INTENTS),
        "input": example["message"],
        "output": example["intent"]
    }

alpaca_data = [to_alpaca_format(ex) for ex in final_dataset]

# Show sample
print("Sample Alpaca-format entry:")
print(json.dumps(alpaca_data[0], indent=2))

# Export as ChatML format (for chat model fine-tuning)
def to_chatml_format(example):
    return {
        "messages": [
            {"role": "system", "content": "You are an intent classifier for customer support. "
             "Respond with only the intent label."},
            {"role": "user", "content": example["message"]},
            {"role": "assistant", "content": example["intent"]}
        ]
    }

chatml_data = [to_chatml_format(ex) for ex in final_dataset]

print("\nSample ChatML-format entry:")
print(json.dumps(chatml_data[0], indent=2))

In [ ]:
# Save to JSONL files
output_dir = Path("output_99")
output_dir.mkdir(exist_ok=True)

# Alpaca format
with open(output_dir / "train_alpaca.jsonl", "w") as f:
    for item in alpaca_data:
        f.write(json.dumps(item) + "\n")

# ChatML format
with open(output_dir / "train_chatml.jsonl", "w") as f:
    for item in chatml_data:
        f.write(json.dumps(item) + "\n")

# Raw JSON
with open(output_dir / "dataset_raw.json", "w") as f:
    json.dump(final_dataset, f, indent=2)

print(f"Exported to {output_dir}/")
print(f"  train_alpaca.jsonl: {len(alpaca_data)} entries")
print(f"  train_chatml.jsonl: {len(chatml_data)} entries")
print(f"  dataset_raw.json:   {len(final_dataset)} entries")

## Step 5 — Verify with Cross-Validation

Use the same LLM to classify a sample to verify label quality.

In [ ]:
# Cross-validate: ask LLM to classify the synthetic data
sample = random.sample(valid, min(6, len(valid)))

print("Cross-validation: LLM re-classifies synthetic examples\n")
correct = 0
for ex in sample:
    msg = llm.invoke([
        SystemMessage(content=f"Classify this customer message into exactly one intent.\n"
                      f"Intents: {', '.join(INTENTS)}\n"
                      f"Reply with ONLY the intent label."),
        HumanMessage(content=ex["message"])
    ])
    predicted = msg.content.strip().lower()
    if "<think>" in predicted:
        predicted = predicted.split("</think>")[-1].strip()
    
    match = ex["intent"].lower() in predicted
    icon = "✅" if match else "❌"
    correct += int(match)
    print(f"  {icon} Label: {ex['intent']:25s} Predicted: {predicted[:30]}")
    print(f"     Message: {ex['message'][:60]}")

print(f"\nLabel agreement: {correct}/{len(sample)} ({100*correct/len(sample):.0f}%)")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Seed examples** | 2 human-written examples per intent |
| **Generation** | LLM generates 5 diverse examples per intent |
| **Quality filters** | Length, duplication, validity checks |
| **Export formats** | Alpaca JSONL + ChatML JSONL |
| **Cross-validation** | LLM re-classifies to verify labels |

## Synthetic Data Pipeline
```
Seed Examples (human) → LLM Generation (diverse)
    → Quality Filter → Export (JSONL)
    → Cross-Validate → Fine-Tune Student Model
```

## Extending This Project
- Use GPT-4/Claude as teacher, fine-tune Llama/Phi as student
- Generate harder edge cases (multi-intent, sarcasm)
- Add human review step between generation and training
- Scale to 10,000+ examples with batched generation
- Compare fine-tuned model accuracy with/without synthetic data